In [137]:
import pandas as pd
scores = pd.read_csv("results_scored.csv")


## Missing result Imputation and Score Rellocation
Algorithm
----
    1. groupby 'Query' and 'Rank"
    2. check if all four methods are recorded
        - 'BERT', 'SBERT', 'TF-IDF + GloVe', 'TF-IDF'
        - record the number of missing methods
    3. if there is at least one method not being represented
        1. add rows for the missing methods with giving the score as 1
        2. increase the other method's score with the number of previously missing methods
        3. ensure thatno method has a score higher than 3
    5. save to a new Dataframe 

In [138]:
import pandas as pd

EXPECTED_METHODS = {
    "BERT",
    "SBERT",
    "TF-IDF + GloVe",
    "TF-IDF"
}

new_groups = []
augmented_scores = pd.DataFrame()

for (query, rank), group in scores.groupby(["Query", "Rank"]):

    group = group.copy()

    # Mark original rows
    group["Imputed"] = False

    present_methods = set(group["Method"])
    missing_methods = EXPECTED_METHODS - present_methods
    n_missing = len(missing_methods)

    if n_missing > 0:

        # Increase existing scores
        group["Score"] = (group["Score"] + n_missing).clip(upper=3)

        for method in missing_methods:

            new_row = {col: pd.NA for col in scores.columns}

            # Required fields
            new_row["Query"] = query
            new_row["Rank"] = rank
            new_row["Method"] = method
            new_row["Score"] = 0

            # Explicitly null out retrieval data
            new_row["Similarity Score (%)"] = pd.NA
            new_row["Text"] = pd.NA
            new_row["Psalm Num"] = pd.NA
            new_row["Verse"] = pd.NA
            new_row["Scored"] = pd.NA
            new_row["Letter"] = pd.NA

            new_row["Imputed"] = True

            group = pd.concat(
                [group, pd.DataFrame([new_row])],
                ignore_index=True
            )

    new_groups.append(group)

augmented_scores = pd.concat(new_groups, ignore_index=True)

# Verification
print(
    augmented_scores
    .groupby(["Query", "Rank"])["Method"]
    .nunique()
    .value_counts()
)

print(
    augmented_scores["Imputed"]
    .value_counts()
)

/var/folders/g5/8ngtx_f52pz3qhgkcqr072_m0000gn/T/ipykernel_37403/2017466563.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  group = pd.concat(
/var/folders/g5/8ngtx_f52pz3qhgkcqr072_m0000gn/T/ipykernel_37403/2017466563.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  group = pd.concat(
/var/folders/g5/8ngtx_f52pz3qhgkcqr072_m0000gn/T/ipykernel_37403/2017466563.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, t

Method
4    420
Name: count, dtype: int64
Imputed
False    10783
True        70
Name: count, dtype: int64


/var/folders/g5/8ngtx_f52pz3qhgkcqr072_m0000gn/T/ipykernel_37403/2017466563.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  group = pd.concat(
/var/folders/g5/8ngtx_f52pz3qhgkcqr072_m0000gn/T/ipykernel_37403/2017466563.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  group = pd.concat(
/var/folders/g5/8ngtx_f52pz3qhgkcqr072_m0000gn/T/ipykernel_37403/2017466563.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, t

In [139]:
augmented_scores[augmented_scores["Query"] == 'struggle']

,Query,Method,Rank,Similarity Score (%),Text,Psalm Num,Verse,Scored,Letter,Score,Imputed
10079,struggle,BERT,1,35.08,Psalter,42,"Judge me, God, and give judgment in my cause a...",False,A,1,False
10080,struggle,SBERT,1,96.24,Psalter,150,Praise God in His holy ones; praise Him in the...,False,B,2,False
10081,struggle,TF-IDF,1,0.00,Psalter,150,Praise God in His holy ones; praise Him in the...,False,C,3,False
10082,struggle,BERT,1,35.08,Psalter,42,"Judge me, God, and give judgment in my cause a...",True,A,3,False
10083,struggle,SBERT,1,96.24,Psalter,150,Praise God in His holy ones; praise Him in the...,True,B,1,False
...,...,...,...,...,...,...,...,...,...,...,...
10323,struggle,SBERT,5,95.96,Psalter,146,The Lord doth build up Jerusalem; He shall gat...,True,B,3,False
10324,struggle,BERT,5,30.93,Psalter,1,Blessed is the man that hath not walked in the...,True,A,3,False
10325,struggle,SBERT,5,95.96,Psalter,146,The Lord doth build up Jerusalem; He shall gat...,True,B,2,False
10326,struggle,TF-IDF + GloVe,5,NaN,NaN,<NA>,NaN,<NA>,NaN,0,True


In [140]:
query = "struggle"

orig = scores[scores["Query"] == query]
aug = augmented_scores[augmented_scores["Query"] == query]

print(f"Query: '{query}'")
print("-" * 40)
print(f"Original rows:              {len(orig)}")
print(f"Original unique rows:       {len(orig.drop_duplicates())}")
print(f"Original duplicates:        {len(orig) - len(orig.drop_duplicates())}")
print()
print(f"Augmented rows:             {len(aug)}")
print(f"Augmented unique rows:      {len(aug.drop_duplicates())}")
print(f"Augmented duplicates:       {len(aug) - len(aug.drop_duplicates())}")

Query: 'struggle'
----------------------------------------
Original rows:              240
Original unique rows:       42
Original duplicates:        198

Augmented rows:             249
Augmented unique rows:      51
Augmented duplicates:       198


In [141]:
# Saving new Data as scores
scores = augmented_scores

In [142]:
scores.shape

(10853, 11)

In [143]:
scores = scores.drop_duplicates()
scores.shape

(4911, 11)

In [144]:
duplicates = scores[
    scores.duplicated(
        subset=['Query', 'Method', 'Rank'],
        keep=False
    )
].sort_values(['Query', 'Method', 'Rank'])


In [145]:
scores = scores.drop_duplicates(
    subset=['Query', 'Method', 'Rank'],
    keep='first'
)

In [146]:
scores.shape

# This is what we needed

(1680, 11)

-  `human_scores` - is the results from human annotators
- `human_seen_llm_results` - is the results scored by the llm in the pairwise comparisons 

In [147]:
human_scores = pd.read_csv("human_scores.csv")
human_scores

,Unnamed: 0,Query,Query Category,Method,Similarity Score (%),numbered_result,Text,Psalm Num,Verse,User,Score
0,0,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,26.10,1,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9
1,1,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,25.67,2,Bible,4,For the End in psalms an ode by David You hear...,caden,6
2,2,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,21.90,3,Bible,31,By David concerning understanding Blessed are ...,caden,3
3,3,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.33,4,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10
4,4,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.12,5,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7
...,...,...,...,...,...,...,...,...,...,...,...
855,941,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,7.08,4,Psalter,61,Shall not my soul be subject unto God? for fro...,p09,8
856,942,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,7.08,4,Psalter,61,Shall not my soul be subject unto God? for fro...,p03,1
857,943,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he may sin, saith withi...",p10,2
858,944,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he may sin, saith withi...",p05,7


In [148]:
human_scores = human_scores.rename(columns={'numbered_result':'Rank'})

In [149]:
human_seen_results = pd.read_csv("results_from_humans_scored.csv")
#human_seen_results

In [150]:
human_seen_llm_results = human_seen_results.drop_duplicates(
    subset=['Query', 'Method', 'Rank'],
    keep='first'
)

human_seen_llm_results = human_seen_llm_results.sort_values(
    by='Query'
)

human_seen_llm_results

,Query,Method,Rank,Similarity Score (%),Text,Psalm Num,Verse,Scored,Letter,Score
156,Create in me a clean heart,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,False,A,1
157,Create in me a clean heart,BERT,4,68.66,Psalter,43,"We have heard with our ears, O God, for our fa...",False,B,3
158,Create in me a clean heart,SBERT,4,96.07,Psalter,147,"Praise the Lord, O Jerusalem; praise thy God, ...",False,C,0
159,Create in me a clean heart,TFIDF,4,12.53,Bible,50,For the End a psalm by David 2when Nathan the ...,False,D,2
105,Create in me a clean heart,TFIDF,2,14.21,Psalter,54,"Give ear to my prayer, O God, and despise not ...",False,D,3
...,...,...,...,...,...,...,...,...,...,...
249,protection from enemies,SBERT,3,96.26,Bible,11,For the End concerning the eighth a psalm by D...,False,C,1
248,protection from enemies,BERT,3,99.57,Bible,71,For Solomon OGod give Your judgments to the Ki...,False,B,2
247,protection from enemies,TFIDF_GLoVe,3,23.85,Psalter,86,His foundations are in the holy mountains. The...,False,A,3
189,protection from enemies,TFIDF,1,10.10,Bible,70,By David of the sons of Jonadab and the first ...,False,D,0


In [151]:
common_queries = list(
    set(human_scores['Query'].unique()) &
    set(human_seen_results['Query'].unique())
)
common_queries

['mercy',
 'Have mercy on me, O God, have mercy on me. For my soul trusts in Thee, and in the shadow of Thy wings will I hope, until iniquity pass away.',
 'Create in me a clean heart',
 'Rejoice, O ye heavens, sound the trumpets, ye foundation of the earth, thunder forth gladness, O ye mountains: for behold, Emmanuel to the Cross our sins, and the Giver of Life hath slain death, raising up Adam; for He loveth mankind.',
 'How does the psalmist express trust in God while surrounded by fear and uncertainty?',
 'For the Peace of the world',
 'praise in times of suffering',
 'prayer',
 'protection from enemies',
 'Verses where the psalmist remembers past deliverance and uses it to find hope in present trials.',
 'The Lord is my shepherd']

In [152]:
human_scores[
    human_scores.duplicated(subset=keys, keep=False)
].sort_values(keys)

,Unnamed: 0,Query,Query Category,Method,Similarity Score (%),Rank,Text,Psalm Num,Verse,User,Score
5,6,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.89,1,Bible,18,For the End a psalm by David The heavens decla...,caden,4
230,259,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.89,1,Bible,18,For the End a psalm by David The heavens decla...,p03,1
231,260,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.89,1,Bible,18,For the End a psalm by David The heavens decla...,p06,8
232,261,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.89,1,Bible,18,For the End a psalm by David The heavens decla...,p02,8
6,7,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.06,2,Psalter,34,"Judge Thou, O Lord, them that do me injustice;...",caden,8
...,...,...,...,...,...,...,...,...,...,...,...
631,696,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,22.61,4,Psalter,28,"Bring unto the Lord, O ye sons of God, bring u...",p08,7
139,152,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,18.97,5,Bible,151,1This is a psalm written with Davids own hand ...,caden,6
632,697,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,18.97,5,Bible,151,1This is a psalm written with Davids own hand ...,p02,9
633,698,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,18.97,5,Bible,151,1This is a psalm written with Davids own hand ...,p01,5


In [153]:
keys = ['Query', 'Method', 'Rank']

# Make sure there is only one LLM score per result
llm_scores = human_seen_llm_results[
    keys + ['Score']
].drop_duplicates(
    subset=keys,
    keep='first'
)

# Add LLM score to the human scores
human_scores_combined = human_scores.merge(
    llm_scores,
    on=keys,
    how='left',
    validate='many_to_one'
)

# Rename the LLM score
human_scores_combined = human_scores_combined.rename(
    columns={'Score': 'LLM_Score'}
)

print("Human scores:", len(human_scores))
print("Combined:", len(human_scores_combined))

Human scores: 860
Combined: 860


In [154]:
human_scores.groupby(keys).size().value_counts().sort_index()

4    215
Name: count, dtype: int64

## Looking at overlap

In [155]:
llm_scores

,Query,Method,Rank,Score
156,Create in me a clean heart,TFIDF_GLoVe,4,1
157,Create in me a clean heart,BERT,4,3
158,Create in me a clean heart,SBERT,4,0
159,Create in me a clean heart,TFIDF,4,2
105,Create in me a clean heart,TFIDF,2,3
...,...,...,...,...
249,protection from enemies,SBERT,3,1
248,protection from enemies,BERT,3,2
247,protection from enemies,TFIDF_GLoVe,3,3
189,protection from enemies,TFIDF,1,0


In [156]:
human_scores

,Unnamed: 0,Query,Query Category,Method,Similarity Score (%),Rank,Text,Psalm Num,Verse,User,Score
0,0,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,26.10,1,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9
1,1,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,25.67,2,Bible,4,For the End in psalms an ode by David You hear...,caden,6
2,2,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,21.90,3,Bible,31,By David concerning understanding Blessed are ...,caden,3
3,3,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.33,4,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10
4,4,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.12,5,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7
...,...,...,...,...,...,...,...,...,...,...,...
855,941,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,7.08,4,Psalter,61,Shall not my soul be subject unto God? for fro...,p09,8
856,942,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,7.08,4,Psalter,61,Shall not my soul be subject unto God? for fro...,p03,1
857,943,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he may sin, saith withi...",p10,2
858,944,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he may sin, saith withi...",p05,7


In [157]:
import numpy as np 
common = human_scores.merge(
    llm_scores,
    on=['Query', 'Method', 'Rank'],
    how='inner'
)

common = common.rename(columns={'Score_y': "llm_score"})
# common

In [158]:
def score_10_to_3(score):
    if score <= 2:
        return 0
    elif score <= 5:
        return 1
    elif score <= 7:
        return 2
    else:
        return 3

In [159]:
common['score_x_condenced'] = common['Score_x'].apply(score_10_to_3)
common.columns

Index(['Unnamed: 0', 'Query', 'Query Category', 'Method',
       'Similarity Score (%)', 'Rank', 'Text', 'Psalm Num', 'Verse', 'User',
       'Score_x', 'llm_score', 'score_x_condenced'],
      dtype='object')

In [160]:
human_wide = (
    common
    .pivot_table(
        index=['Query', 'Query Category', 'Method', 'Similarity Score (%)', 'Rank', 'Text', 'Psalm Num', 'Verse'],
        columns='User',
        values='score_x_condenced',
        aggfunc='first'
    )
    .reset_index()
)

In [161]:
human_wide

User,Query,Query Category,Method,Similarity Score (%),Rank,Text,Psalm Num,Verse,caden,p01,...,p04,p05,p06,p07,p08,p09,p10,p13,p16,p17
0,Create in me a clean heart,Phrase/Exact Match Queries,BERT,68.31,5,Bible,147,Alleluia of Aggeus and Zacharias Praise the Lo...,1.0,NaN,...,NaN,2.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,Create in me a clean heart,Phrase/Exact Match Queries,BERT,68.66,4,Psalter,43,"We have heard with our ears, O God, for our fa...",0.0,1.0,...,NaN,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Create in me a clean heart,Phrase/Exact Match Queries,BERT,68.69,3,Psalter,49,"The God of gods, even the Lord, hath spoken, a...",1.0,NaN,...,NaN,3.0,2.0,NaN,NaN,NaN,NaN,3.0,NaN,NaN
3,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.06,2,Psalter,34,"Judge Thou, O Lord, them that do me injustice;...",3.0,NaN,...,NaN,NaN,1.0,NaN,2.0,NaN,1.0,NaN,NaN,NaN
4,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.89,1,Bible,18,For the End a psalm by David The heavens decla...,1.0,NaN,...,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,18.97,5,Bible,151,1This is a psalm written with Davids own hand ...,2.0,1.0,...,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
189,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,22.61,4,Psalter,28,"Bring unto the Lord, O ye sons of God, bring u...",1.0,NaN,...,NaN,NaN,0.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN
190,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,23.85,3,Psalter,86,His foundations are in the holy mountains. The...,1.0,NaN,...,1.0,NaN,0.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN
191,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,25.01,2,Bible,28,A psalm by David the final day of the Feast of...,3.0,1.0,...,NaN,NaN,0.0,NaN,3.0,NaN,NaN,NaN,NaN,NaN


In [165]:
scores = human_wide

In [168]:
scores.to_csv('human_and_llm_scores.csv')

In [164]:
queries = pd.read_csv('queries.csv')

EmptyDataError: No columns to parse from file

In [ ]:
scores["Query Category"] = scores["Query"].map(
    queries.set_index("Query")["Query Category"]
)

scores.head(1)

In [ ]:
queries['Query Category'].value_counts().sort_values(ascending=False)

In [ ]:
# Adding the 5 results per the four methods for each query query to make sure it 
# matches up with the data in the scores dataframe

(queries['Query Category'].value_counts()*20).sort_values(ascending=False)

In [ ]:
#pivot for each category
pd.pivot_table(
    data=scores,
    index='Query Category',
    values='Score',
    aggfunc='count'
).sort_values('Score', ascending=False)

* we can confirm that alll of the query caetorgiers are properly represented within the data based on the queries used and the corresponding query categories
* by working and adjusting the math for the values counts we can see that all the data is represented correctly

============================================

## Krippendorff Alphs from Human Annotators and LLM

In [169]:
scores = pd.read_csv('human_and_llm_scores.csv')

In [170]:
scores.head()

,Unnamed: 0,Query,Query Category,Method,Similarity Score (%),Rank,Text,Psalm Num,Verse,caden,...,p04,p05,p06,p07,p08,p09,p10,p13,p16,p17
0,0,Create in me a clean heart,Phrase/Exact Match Queries,BERT,68.31,5,Bible,147,Alleluia of Aggeus and Zacharias Praise the Lo...,1.0,...,NaN,2.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,1,Create in me a clean heart,Phrase/Exact Match Queries,BERT,68.66,4,Psalter,43,"We have heard with our ears, O God, for our fa...",0.0,...,NaN,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,Create in me a clean heart,Phrase/Exact Match Queries,BERT,68.69,3,Psalter,49,"The God of gods, even the Lord, hath spoken, a...",1.0,...,NaN,3.0,2.0,NaN,NaN,NaN,NaN,3.0,NaN,NaN
3,3,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.06,2,Psalter,34,"Judge Thou, O Lord, them that do me injustice;...",3.0,...,NaN,NaN,1.0,NaN,2.0,NaN,1.0,NaN,NaN,NaN
4,4,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.89,1,Bible,18,For the End a psalm by David The heavens decla...,1.0,...,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [172]:
%pip install krippendorff

You should consider upgrading via the '/usr/local/bin/python3.9 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [174]:
import krippendorff

scores.columns

Index(['Unnamed: 0', 'Query', 'Query Category', 'Method',
       'Similarity Score (%)', 'Rank', 'Text', 'Psalm Num', 'Verse', 'caden',
       'p01', 'p02', 'p03', 'p04', 'p05', 'p06', 'p07', 'p08', 'p09', 'p10',
       'p13', 'p16', 'p17'],
      dtype='object')

In [ ]:
annotator_cols = ['p01', 'p02', 'p03', 'p06']

data = human_wide[annotator_cols].to_numpy().T

alpha = krippendorff.alpha(
    reliability_data=data,
    level_of_measurement='ordinal'
)

print(f"Krippendorff's Alpha: {alpha:.4f}")

In [ ]:
print(human_seen_results.columns.tolist())

In [ ]:
human_scores = human_scores.rename(columns={'numbered_result':'Rank'})

In [ ]:
keys = [
    'Query',
    'Method',
    'Rank',
    'Similarity Score (%)',
    'Text',
    'Psalm Num',
    'Verse'
]

# Turn the repeated human evaluations into 4 score columns
human_scores = (
    human_scores
    .pivot_table(
        index=keys,
        columns='User',
        values='Score',
        aggfunc='first'
    )
    .reset_index()
)

human_scores.columns.name = None

In [ ]:
human_scores

In [ ]:
human_scores['Similarity Score (%)'] = pd.to_numeric(
    human_scores['Similarity Score (%)'],
    errors='coerce'
)

scores['Similarity Score (%)'] = pd.to_numeric(
    scores['Similarity Score (%)'],
    errors='coerce'
)

In [ ]:
pd.merge(scores, human_scores, how='inner')

In [ ]:
keys = [
    'Query',
    'Method',
    'Rank',
    'Similarity Score (%)',
    'Text',
    'Psalm Num',
    'Verse'
]

human_scores = human_scores.merge(
    filtered_scores[keys + ['Score']],
    on=keys,
    how='left',
    suffixes=('', '_LLM')
)

human_scores.head(50)